In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import time
import matplotlib.pyplot as plt

import sys, os
sys.path.append(os.path.abspath(".."))

from src.core import *
import src.viz as viz
import src.io as io

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device ->", device)

cell_cfg = CellConfig(hidden_channels=8, visible_channels=1, alive_threshold=0.05)
perc_cfg = PerceptionConfig(kernel_radius=1, channel_groups=3)
upd_cfg = UpdateConfig(hidden_dim=64, stochastic_update=False, fire_rate=0.5)
grid_cfg = GridConfig(size=(32, 32, 32))

model = Grid3D(cell_cfg, perc_cfg, upd_cfg, grid_cfg).to(device)

In [ ]:
filepath = os.path.abspath("../assets/donut/model.obj")
target = io.obj_to_tensor(filepath=filepath, grid_size=(32, 32, 32), mode="alpha", device=device)

viz.show_slice_alpha_mpl(target,visible_channels=1, title="Target Slice", axis=1, idx=15)
viz.show_volume_alpha_mpl(target, title="Target Volume", point_size=100)
viz.show_volume_alpha_pv(target, notebook=True)

In [ ]:
print("stability run:")
state = model.seed_center(batch_size=1, device=device)
state += 1e-3 * torch.randn_like(state)
with torch.no_grad():
    for i in range(8):
        state = model(state, steps=1)
        print(f"step {i:02d} mean={state.mean().item():+.6f} std={state.std().item():.6f}")

In [ ]:
print("\nstarting multi-step training")
optim = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
losses = []
iters = 2000
n_steps = 8
log_interval = 200

start = time.time()
for it in range(1, iters+1):
    state = model.seed_center(batch_size=1, device=device)
    state += 0.02 * torch.randn_like(state)
    optim.zero_grad()
    loss = 0.0
    for step in range(n_steps):
        state = model(state, steps=1)
        alpha = state[:, -1:, ...]
        loss += F.mse_loss(alpha, target)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optim.step()
    losses.append(loss.item())
    
    if it % log_interval == 0 or it==1:
        print(f"Iter {it:05d}, Loss={loss.item():.6f}, Mean Alpha={alpha.mean().item():.4f}")
        viz.show_slice_alpha_comparison_mpl(state, target, axis=1, idx=15, visible_channels=1)
    if loss.item() < 0.095:
        print("Best result")
        print(f"Iter {it:05d}, Loss={loss.item():.6f}, Mean Alpha={alpha.mean().item():.4f}")
        viz.show_slice_alpha_comparison_mpl(state, target, axis=1, idx=15, visible_channels=1)
        break

print("training took", time.time() - start, "s")

In [ ]:
with torch.no_grad():
    state = model.seed_center(batch_size=1, device=device)
    state += 1e-3 * torch.randn_like(state)
    state = model(state, steps=8)
    alpha = state[:, -1:, ...].cpu().numpy().squeeze()

thresh = 0.2
xs, ys, zs = np.nonzero(alpha > thresh)
print("final alive voxels:", len(xs))
viz.show_volume_alpha_comparison_mpl(state, target, point_size=12, visible_channels=1)
viz.show_volume_alpha_comparison_pv(state, target, visible_channels=1, notebook=True)

In [ ]:
plt.figure()
plt.plot(losses)
plt.yscale("log")
plt.title("training loss (log)")
plt.xlabel("iter")
plt.ylabel("mse")
plt.show()